In [ ]:
from langgraph.graph import StateGraph, START, MessageState
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI()

### Without any Checkpointer

In [ ]:
def call_model(state: MessageState):
    response = model.invoke(state['messages'])
    return {"messages": [response]}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [ ]:
graph = builder.compile()
graph

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Koyel."}]})

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})

### With Checkpointer

In [ ]:
def call_model(state: MessageState):
    response = model.invoke(state['messages'])
    return {"messages": [response]}

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer = checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "thread-1"}}

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Koyel."}]}, config)

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)

In [ ]:
snap = graph.get_state(config)
vals = snap.values
for m in vals.get("messages", []):
    print("-", type(m).__name__, ";", m.content)

### Checkpointer with Multiple Thread (Memory of one thread is not accessible tp other threads) 

In [ ]:
config2 = {"configurable": {"thread_id": "thread-2"}}

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config2)

In [ ]:
snap = graph.get_state(config2)
vals = snap.values
for m in vals.get("messages", []):
    print("-", type(m).__name__, ";", m.content)

### Till now we worked in RAM, on restart, all old memory will be lost. In production setup, INMemorySaver will not work bcz of this reason, we should use any database for thsi purpose.

### 